# Experiment Performance Analysis

Visualizes pass/fail distributions across Langfuse dataset runs (experiments).

## Prerequisites
- Langfuse running (default: `http://localhost:3000`)
- `.env` at workspace root with `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and optionally `EVAL_LANGFUSE_BASE_URL`
- At least one eval run pushed to Langfuse via `uv run eval-run --langfuse`

## Usage
Change `DATASET_NAME` below to analyze a different skill's experiments.
Run names correspond to `--run-name` values passed to `eval-run` (e.g. `scaffold-baseline`, `v2-prompt`).

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load env vars FIRST, then override LANGFUSE_BASE_URL before importing langfuse
load_dotenv(Path("../.env"))
os.environ["LANGFUSE_BASE_URL"] = (
    os.environ.get("EVAL_LANGFUSE_BASE_URL", "").strip() or "http://localhost:3000"
)

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from langfuse import Langfuse

# --- Configuration ---
DATASET_NAME = "evals/Meeting Summariser"
# ---------------------

In [2]:
langfuse = Langfuse()
print(f"Connected to Langfuse at {os.environ['LANGFUSE_BASE_URL']}")

Connected to Langfuse at http://localhost:3000


In [3]:
# Fetch all runs for the dataset
runs_response = langfuse.get_dataset_runs(dataset_name=DATASET_NAME)
runs = runs_response.data
print(f"Found {len(runs)} experiment run(s) for '{DATASET_NAME}'")
for r in runs:
    print(f"  - {r.name} (created: {r.created_at})")

Found 2 experiment run(s) for 'evals/Meeting Summariser'
  - 2_threshold_updated (created: 2026-05-21 01:28:33.606000+00:00)
  - 1.baseline (created: 2026-05-15 07:59:59.351000+00:00)


In [4]:
# Fetch verdict scores for each run
records = []

for run in runs:
    run_detail = langfuse.get_dataset_run(
        dataset_name=DATASET_NAME, run_name=run.name
    )
    for item in run_detail.dataset_run_items:
        scores = langfuse.api.scores.get_many(
            trace_id=item.trace_id, name="verdict", limit=1
        )
        verdict = None
        if scores.data:
            verdict = bool(scores.data[0].value)
        records.append({
            "run_name": run.name,
            "created_at": run.created_at,
            "trace_id": item.trace_id,
            "dataset_item_id": item.dataset_item_id,
            "verdict": verdict,
        })

df = pd.DataFrame(records)
print(f"Collected {len(df)} trace verdicts across {df['run_name'].nunique()} runs")
df.head()

Collected 16 trace verdicts across 2 runs


,run_name,created_at,trace_id,dataset_item_id,verdict
0,2_threshold_updated,2026-05-21 01:28:33.606000+00:00,6f7b9ab33256c7008ebbc8dbb9ce75ed,37bb98d2-1b92-5b82-b95f-7ecfc1367fd5,False
1,2_threshold_updated,2026-05-21 01:28:33.606000+00:00,c0dd063e173191c8cc63e940b2f2e543,ba51e19b-b378-5ad1-875b-615d15eb127c,True
2,2_threshold_updated,2026-05-21 01:28:33.606000+00:00,dab4c7065a47fbd264980ee306d20d9c,ea58ad42-3fa2-5f86-a942-6651952440bf,True
3,2_threshold_updated,2026-05-21 01:28:33.606000+00:00,7b2ff7705d8061235bf3efe385071efa,62c785b1-23cb-52e0-a8ef-ee747ca93a0e,True
4,2_threshold_updated,2026-05-21 01:28:33.606000+00:00,c4d9780d4c3d131f94801f3bc4411014,fae147eb-5959-57bf-91f9-18e632a75a3a,True


In [5]:
# Aggregate pass/fail per run
run_order = (
    df.groupby("run_name")["created_at"]
    .first()
    .sort_values()
    .index.tolist()
)

summary = (
    df.groupby("run_name")["verdict"]
    .agg(total="count", passed=lambda x: x.sum(), failed=lambda x: (~x).sum())
    .reindex(run_order)
)
summary["pass_rate"] = (summary["passed"] / summary["total"] * 100).round(1)
summary

,total,passed,failed,pass_rate
run_name,,,,
1.baseline,8,6,2,75.0
2_threshold_updated,8,6,2,75.0


In [6]:
# Visualization: Pass Rate Trend + Pass/Fail Distribution
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Pass Rate by Experiment Run", "Pass/Fail Distribution"),
    horizontal_spacing=0.12,
)

# Left panel: Pass rate trend line
fig.add_trace(
    go.Scatter(
        x=summary.index,
        y=summary["pass_rate"],
        mode="lines+markers+text",
        text=[f"{v}%" for v in summary["pass_rate"]],
        textposition="top center",
        textfont=dict(color="#2e7d32", size=12, family="Arial Black"),
        line=dict(color="#4caf50", width=3),
        marker=dict(size=10, color="#2e7d32"),
        showlegend=False,
    ),
    row=1, col=1,
)

# Right panel: Grouped bar chart
fig.add_trace(
    go.Bar(
        x=summary.index,
        y=summary["passed"],
        name="Pass",
        marker_color="#8bc34a",
        text=summary["passed"].astype(int),
        textposition="outside",
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Bar(
        x=summary.index,
        y=summary["failed"],
        name="Fail",
        marker_color="#ef9a9a",
        text=summary["failed"].astype(int),
        textposition="outside",
    ),
    row=1, col=2,
)

fig.update_yaxes(title_text="Pass Rate (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_layout(
    title_text=f"{DATASET_NAME.split('/')[-1]} Performance Analysis",
    title_x=0.5,
    title_font_size=18,
    barmode="group",
    height=450,
    width=1100,
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.75),
    template="plotly_white",
)
fig.show()

In [7]:
# Per-Case Heatmap: identify which cases regress across experiments
if df["run_name"].nunique() > 1:
    pivot = df.pivot_table(
        index="dataset_item_id",
        columns="run_name",
        values="verdict",
        aggfunc="first",
    )
    pivot = pivot.reindex(columns=run_order)

    # Map item IDs to short labels
    item_labels = [f"item_{i+1}" for i in range(len(pivot))]

    heatmap = go.Figure(
        data=go.Heatmap(
            z=pivot.values.astype(float),
            x=pivot.columns.tolist(),
            y=item_labels,
            colorscale=[[0, "#ef9a9a"], [1, "#8bc34a"]],
            showscale=False,
            zmin=0,
            zmax=1,
            text=pivot.applymap(lambda v: "PASS" if v else "FAIL").values,
            texttemplate="%{text}",
            hovertemplate="Run: %{x}<br>Item: %{y}<br>Verdict: %{text}<extra></extra>",
        )
    )
    heatmap.update_layout(
        title="Per-Case Verdicts Across Experiments",
        title_x=0.5,
        xaxis_title="Experiment Run",
        yaxis_title="Dataset Item",
        height=max(300, len(pivot) * 35),
        width=900,
        template="plotly_white",
    )
    heatmap.show()
else:
    print("Heatmap requires at least 2 experiment runs for comparison.")

AttributeError: 'DataFrame' object has no attribute 'applymap'

In [ ]:
import re
from IPython.display import display


def failure_drilldown(
    df,
    langfuse,
    filter_run=None,
    show_query=True,
    show_actual=False,
    show_expected=False,
):
    """Render a styled table of failed cases with metric details.

    Args:
        df: DataFrame with trace verdicts (from cell 5).
        langfuse: Langfuse client instance.
        filter_run: Optional run name to filter to.
        show_query: Include user_query column (trace input).
        show_actual: Include actual_output column.
        show_expected: Include expected_output column.
    """
    failed_traces = df[df["verdict"] == False]
    if filter_run:
        failed_traces = failed_traces[failed_traces["run_name"] == filter_run]

    if failed_traces.empty:
        print("No failed cases found.")
        return

    failure_details = []
    for _, row in failed_traces.iterrows():
        all_scores = langfuse.api.scores.get_many(trace_id=row["trace_id"], limit=50)
        trace = langfuse.api.trace.get(row["trace_id"])
        case_key = trace.metadata.get("case_key", "unknown") if trace.metadata else "unknown"
        case_name = trace.metadata.get("case_name", "") if trace.metadata else ""

        user_query = ""
        if show_query:
            user_query = trace.input or ""
        actual_output = ""
        if show_actual:
            actual_output = trace.output or ""
        expected_output = ""
        if show_expected:
            try:
                dataset_item = langfuse.api.dataset_items.get(row["dataset_item_id"])
                expected_output = dataset_item.expected_output or ""
            except Exception:
                expected_output = "(could not fetch)"

        for score in all_scores.data:
            if score.name == "verdict":
                continue
            is_failed = False
            if score.data_type == "BOOLEAN" and score.value == 0:
                is_failed = True
            elif score.data_type == "NUMERIC" and score.comment:
                if "passed=False" in score.comment:
                    is_failed = True

            if not is_failed:
                continue

            reasons_text = ""
            if score.comment and "\n\n" in score.comment:
                reasons_text = score.comment.split("\n\n", 1)[1]

            run_reasons = re.split(r"\n\n(?=\[Run \d+\])", reasons_text) if reasons_text else []

            entries = []
            if run_reasons:
                for run_reason in run_reasons:
                    entries.append(run_reason.strip())
            else:
                entries.append("(no reason recorded)")

            for reason in entries:
                record = {
                    "run": row["run_name"],
                    "case": f"{case_key}: {case_name}",
                    "metric": score.name,
                    "score": round(score.value, 3) if score.value is not None else None,
                    "reason": reason,
                }
                if show_query:
                    record["user_query"] = user_query
                if show_actual:
                    record["actual_output"] = actual_output
                if show_expected:
                    record["expected_output"] = expected_output
                failure_details.append(record)

    if not failure_details:
        print("Failed cases found but no individual metric failure details available.")
        return

    failures_df = pd.DataFrame(failure_details)
    print(f"{len(failures_df)} reason(s) across {failed_traces.shape[0]} failed case(s)\n")

    wide_cols = []
    if show_query:
        wide_cols.append("user_query")
    if show_actual:
        wide_cols.append("actual_output")
    if show_expected:
        wide_cols.append("expected_output")

    styled = failures_df.style.set_properties(**{
        "text-align": "left",
        "vertical-align": "top",
    })
    if wide_cols:
        styled = styled.set_properties(subset=wide_cols, **{
            "white-space": "pre-wrap",
            "font-family": "monospace",
            "font-size": "12px",
            "max-width": "500px",
            "overflow": "auto",
        })
    styled = (
        styled
        .set_properties(subset=["reason"], **{
            "white-space": "pre-wrap",
            "max-width": "300px",
        })
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left"), ("vertical-align", "bottom")]},
        ])
        .hide(axis="index")
    )
    display(styled)

In [20]:
failure_drilldown(df, langfuse, show_actual=True, show_expected=True)

2 reason(s) across 2 failed case(s)



run,case,metric,score,reason,user_query,actual_output,expected_output
1.baseline,case_8: Budget catch-up notes → insight question and email workflow,Task Completion,0.567000,[Run 1] The actual outcome involved creating a draft email related to the budget meeting but did not transform the meeting notes into a professional meeting record as required. The focus was on communication rather than documentation.,Turn these budget meeting notes into a proper meeting record please.,"Here’s a professional and friendly email draft for the finance team: --- **Subject:** Budget Update & Q3 Actuals Submission Hi Finance Team, We had a quick budget review with Tom and Rachel earlier this week. Cloud infrastructure costs are currently tracking about 12% over budget for the quarter, largely due to higher-than-expected storage expenses from our new data lake. To address this, we’ve agreed to migrate cold data to the archive tier, which should save approximately $8k per month. Tom will raise the change request by Friday. As a reminder, we understand that Q3 actuals are required by **28 February**. Rachel will ensure these figures are compiled and sent through ahead of the deadline. Thanks for your support, and please let us know if you need any additional detail on the budget or planned cost-saving measures. Best regards, [Your Name] --- Do you want me to also prepare a **shorter, bullet-point version** for quick internal updates?","A complete workflow: skill invocation, detailed structured summary, an insight answer about the budget overrun percentage, and a formatted email to the finance team."
1.baseline,case_3: Design review transcript → detailed summary with decisions,Decision Captured,0.667000,"[Run 1] The output does not provide a summary of the design review meeting or clearly state the decision to proceed with Option C. Instead, it asks for the detail level desired, which does not align with the requirement to summarize the meeting and highlight the decision made.","Summarise this design review meeting. I need a full structured summary with overview, participants, discussion items, decisions, and action items.","Here’s your **Detailed** structured summary of the design review meeting: --- **Overview** The team reviewed three design options for the customer portal dashboard, agreed on Option C, and set follow-up actions for accessibility and development. **Participants** Olivia Hart Daniel Nguyen Karen Brooks Raj Mehta Fiona Clark *(Note: Participant list reflects only those who spoke in the meeting.)* **Discussion Items** ``` Discussion Item 1: Design Options - Daniel presented three concepts: card-based, table view, and hybrid layout. - Option C combines summary cards with a detailed table. Discussion Item 2: Product Perspective - Karen preferred Option C based on user research showing need for snapshot and detail. - Stakeholder feedback on hybrid approach was positive. Discussion Item 3: Development Feasibility - Raj confirmed Option C is most feasible within timeline. - Existing card components can be reused; table component requires extension. Discussion Item 4: Accessibility Concerns - Fiona flagged collapsible sections in Option B as problematic for screen readers. - Colour contrast on status badges in all options fails WCAG AA; Daniel to fix. Discussion Item 5: Stakeholder Confirmation - Karen to confirm Option C with executive sponsor by Thursday. - Positive preliminary feedback already received. Discussion Item 6: Technical Spike - Raj to start proof-of-concept for table component this sprint. - Delivery expected by next Monday. Discussion Item 7: Accessibility Audit - Fiona requested dedicated audit before design phase ends. - Olivia to schedule for first week of April. ``` **Action Items** ``` 1. Fix Colour Contrast (Daniel Nguyen): Increase status badge contrast to meet WCAG AA by tomorrow. 2. Confirm Stakeholder Approval (Karen Brooks): Validate Option C with executive sponsor by Thursday. 3. Table Component POC (Raj Mehta)